In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib


# Round 3 Reproduction

**Purpose.** Reproduce two checks introduced in Round 3: observation streams use the same hidden true state as realized utility, and active information search is evaluated with true-state equal-outcome metrics rather than belief-only equality.

The full searches used 1,200 episodes, 500 VOI samples, common observation streams, and zero prior samples.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print(PROJECT_ROOT)


In [ ]:
RUN = False
EPISODES = 1200
VOI_SAMPLES = 500
OBSERVATIONS_PER_PERSON = 500
GRID_CHUNKS = 48
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
OUTPUT_ROOT = "results/round_03_notebook"
print({"run": RUN, "episodes": EPISODES, "voi_samples": VOI_SAMPLES, "grid_chunks": GRID_CHUNKS})


## Observation-stream invariant

**Test purpose.** Verify true-state identity, the first observations consumed by the simulator, and the correlation between each stream mean and the corresponding hidden need.


In [ ]:
observation_check = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "check_observation_streams.py"),
    "--episodes", "120",
    "--observations-per-person", "500",
    "--min-correlation", "0.95",
    "--output-json", OUTPUT_ROOT + "/observation_stream_check.json",
]
print(" ".join(observation_check))
if RUN:
    subprocess.run(observation_check, cwd=PROJECT_ROOT, check=True)


## Active-search true equal-outcome grids

These grids test whether the RR approximation starts with no prior samples, gathers information, chooses unequal allocations, and approaches equal outcomes in the hidden true state.


In [ ]:
GRID_NAMES = ["active_search_equal_outcome_focused", "active_search_equal_outcome_narrow_followup"]
for grid_name in GRID_NAMES:
    command = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
        "--preset", "server",
        "--sections", "regime_grid",
        "--regime-grid", grid_name,
        "--regime-grid-chunks", str(GRID_CHUNKS),
        "--episodes", str(EPISODES),
        "--voi-samples", str(VOI_SAMPLES),
        "--common-observations", "on",
        "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
        "--max-workers", str(MAX_WORKERS),
        "--output-dir", OUTPUT_ROOT + "/" + grid_name,
    ]
    print(" ".join(command))
    if RUN:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Inspect true-state diagnostics

The primary fields are `true_equal_outcome_rate`, `mean_realized_outcome_gap`, and `closer_to_true_equal_outcome_than_equal_split_rate`. Sample count and allocation distance from 50/50 determine whether a candidate is an active, unequal-allocation strategy.


In [ ]:
import pandas as pd

output_root = PROJECT_ROOT / OUTPUT_ROOT
for path in sorted(output_root.rglob("targeted_regime_behavior_candidates.csv")) if output_root.exists() else []:
    frame = pd.read_csv(path)
    columns = [column for column in ("environment", "candidate_type", "true_equal_outcome_rate", "mean_realized_outcome_gap", "closer_to_true_equal_outcome_than_equal_split_rate", "mean_sample_count", "mean_abs_allocation_from_equal") if column in frame.columns]
    print(path.relative_to(output_root))
    display(frame[columns].head(20))
